# Indoor Scene Change Detection — Training & Evaluation Pipeline


## 1. Environment Setup


In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q
import os, json, shutil, glob, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print("Libraries ready.")


## 7. Validation-Based Model Selection (quick, repeatable check)


In [ ]:
def evaluate_model(model_path, model_name, data_yaml):
    model = YOLO(model_path)
    results = model.val(data=data_yaml)
    p, r = results.box.mp, results.box.mr
    return {
        'model': model_name,
        'mAP50': results.box.map50,
        'mAP50-95': results.box.map,
        'precision': p,
        'recall': r,
        'f1': 2 * p * r / (p + r + 1e-9)
    }

yolo_metrics   = evaluate_model(YOLO_WEIGHTS, 'YOLO11s', '/content/data.yaml')
rtdetr_metrics = evaluate_model(RTDETR_WEIGHTS, 'RT-DETR', '/content/data.yaml')

comparison_df = pd.DataFrame([yolo_metrics, rtdetr_metrics])
comparison_df


In [ ]:
print("VALIDATION-split metrics (quick check only -- see the note in the Section 7 ")
print("markdown above for why there's no chart for this one):")
display(comparison_df)
print()
print("The TEST-split comparison -- and its chart -- are in Section 7b, right below.")


## 7b. Final Detector Evaluation on TEST


In [ ]:
def evaluate_model_on_split(model_path, model_name, data_yaml, split):

    model = YOLO(model_path)
    results = model.val(data=data_yaml, split=split)
    p, r = results.box.mp, results.box.mr
    return {
        'model': model_name, 'split': split,
        'mAP50': results.box.map50, 'mAP50-95': results.box.map,
        'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r + 1e-9),
        'save_dir': str(results.save_dir),

    }

yolo_test_metrics   = evaluate_model_on_split(YOLO_WEIGHTS, 'YOLO11s', '/content/data.yaml', 'test')
rtdetr_test_metrics = evaluate_model_on_split(RTDETR_WEIGHTS, 'RT-DETR', '/content/data.yaml', 'test')

comparison_test_df = pd.DataFrame([yolo_test_metrics, rtdetr_test_metrics])
print("=== FINAL TEST-SET detector metrics (touched exactly once) ===")
comparison_test_df


### 7b-i. Model Comparison Chart -- TEST split

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
comparison_test_df.set_index('model')[['mAP50', 'mAP50-95', 'precision', 'recall', 'f1']].plot(
    kind='bar', ax=ax)
plt.title('YOLO11s vs RT-DETR -- Detection Performance (TEST split, touched once)')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1.05)
plt.legend(loc='lower right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/model_comparison_test.png', dpi=150)
plt.show()

print("This is the chart to use in the report: both detectors compared on the TEST split")
print("they were never trained or threshold-tuned on -- not validation.")


### 7b-ii. Detector Confusion Matrix & PR Curves -- TEST split


In [ ]:
from IPython.display import Image as IPyImage, display as ipy_display

_det_plot_files = ['confusion_matrix.png', 'confusion_matrix_normalized.png',
                    'PR_curve.png', 'F1_curve.png']

for label, metrics in [('YOLO11s', yolo_test_metrics), ('RT-DETR', rtdetr_test_metrics)]:
    print(f"{label} -- TEST split (from {metrics['save_dir']}):")
    shown = False
    for fname in _det_plot_files:
        fpath = os.path.join(metrics['save_dir'], fname)
        if os.path.exists(fpath):
            ipy_display(IPyImage(filename=fpath))
            shown = True
    if not shown:
        print("  (no plot files found in save_dir -- Ultralytics version may differ)")
    print()
